In [4]:
import sys
import os
sys.path.append('/content/DIABETES_130_US')

import torch
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from constants import DATA_FILE, SEED, FEATURE_NAMES, LABEL_NAME
from diabetes_dataset import preprocess_diabetes
from utils import make_run_key, try_load_results, save_results, compute_metrics, aggregate_fold_metrics
from net import DNN, train_kfold as train_kfold_dnn
from TN_net import TN_DNN, train_kfold_TNT
from dGBT_generated import train_kfold_tnt_dgbdt


def train_kfold_xgb_baseline(df, n_splits=5, save_dir=None, force_retrain=False):
    key = make_run_key(model="xgb", splits=n_splits)
    if not force_retrain:
        cached = try_load_results(save_dir, key)
        if cached is not None:
            print(f"[XGBoost] Loaded cached results ({key})")
            return cached

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    labels = df[LABEL_NAME].values

    fold_metrics_list = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(df, labels), start=1):
        fold_train_df = df.iloc[train_idx]
        fold_val_df   = df.iloc[val_idx]

        X_train = fold_train_df[FEATURE_NAMES].values
        y_train = fold_train_df[LABEL_NAME].values.astype(int)
        X_val   = fold_val_df[FEATURE_NAMES].values
        y_val   = fold_val_df[LABEL_NAME].values.astype(int)

        n_pos = y_train.sum()
        scale_pos_weight = (len(y_train) - n_pos) / max(n_pos, 1)

        dtrain = xgb.DMatrix(X_train, label=y_train)
        dval   = xgb.DMatrix(X_val, label=y_val)

        params = {
            "objective": "binary:logistic",
            "max_depth": 6,
            "scale_pos_weight": scale_pos_weight,
            "eval_metric": "auc",
            "verbosity": 0,
            "seed": SEED,
        }
        model = xgb.train(
            params, dtrain,
            num_boost_round=500,
            evals=[(dval, "val")],
            early_stopping_rounds=20,
            verbose_eval=False,
        )

        if save_dir is not None:
            import os; os.makedirs(save_dir, exist_ok=True)
            model.save_model(os.path.join(save_dir, f"{key}__fold{fold}.json"))

        probs = model.predict(dval)
        fold_m = compute_metrics(y_val, probs, verbose=True)
        fold_metrics_list.append(fold_m)
        print(f"Fold {fold} — AUROC: {fold_m['auroc']:.4f}  AUPRC: {fold_m['auprc']:.4f}")

    results = aggregate_fold_metrics(fold_metrics_list)

    if save_dir is not None:
        save_results(save_dir, key, results)

    return results


def _print_model_block(label, results, prefix=""):
    p = prefix
    cm = results.get(f"{p}confusion_matrix_sum")
    print(f"\n  [{label}]")
    print(f"  AUROC:           {results[f'mean_{p}auroc']:.4f} ± {results[f'std_{p}auroc']:.4f}")
    print(f"  AUPRC:           {results[f'mean_{p}auprc']:.4f} ± {results[f'std_{p}auprc']:.4f}")
    print(f"  Accuracy:        {results[f'mean_{p}accuracy']:.4f} ± {results[f'std_{p}accuracy']:.4f}")
    print(f"  Balanced Acc:    {results[f'mean_{p}balanced_accuracy']:.4f} ± {results[f'std_{p}balanced_accuracy']:.4f}")
    print(f"  MCC:             {results[f'mean_{p}mcc']:.4f} ± {results[f'std_{p}mcc']:.4f}")
    print(f"  F1 Macro:        {results[f'mean_{p}f1_macro']:.4f} ± {results[f'std_{p}f1_macro']:.4f}")
    print(f"  Class 0  P/R/F1: {results[f'mean_{p}precision_0']:.4f} / {results[f'mean_{p}recall_0']:.4f} / {results[f'mean_{p}f1_0']:.4f}")
    print(f"  Class 1  P/R/F1: {results[f'mean_{p}precision_1']:.4f} / {results[f'mean_{p}recall_1']:.4f} / {results[f'mean_{p}f1_1']:.4f}")
    if cm:
        tn, fp, fn, tp = cm[0][0], cm[0][1], cm[1][0], cm[1][1]
        print(f"  Confusion Matrix (summed over folds):")
        print(f"             Pred 0    Pred 1")
        print(f"  Actual 0   TN={tn:6d}  FP={fp:6d}")
        print(f"  Actual 1   FN={fn:6d}  TP={tp:6d}")


def print_results(name, results):
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    if "mean_auroc" in results:
        _print_model_block("Model", results, prefix="")
    if "mean_teacher_auroc" in results:
        _print_model_block("Teacher (TN-DNN)", results, prefix="teacher_")
    if "mean_soft_tree_auroc" in results:
        _print_model_block("Student (SDT)", results, prefix="soft_tree_")
    if "mean_dgbdt_auroc" in results:
        _print_model_block("Student (dGBDT)", results, prefix="dgbdt_")
    print(f"{'='*60}")


In [2]:

DATA_FRACTION = None  # e.g. 0.1 to use 10% of data; None to use all
MODELS_DIR    = "saved_models"  # directory to save/load results; None to disable
FORCE_RETRAIN = True           # set True to ignore cached results and retrain

print("Loading and preprocessing data...")
df = preprocess_diabetes(DATA_FILE)
if DATA_FRACTION is not None:
    sss = StratifiedShuffleSplit(n_splits=1, train_size=DATA_FRACTION, random_state=SEED)
    idx, _ = next(sss.split(df, df[LABEL_NAME]))
    df = df.iloc[idx].reset_index(drop=True)
print(f"Dataset shape after preprocessing: {df.shape}")
print(f"Positive rate: {df[LABEL_NAME].mean():.4f}")

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
N_EPOCHS  = 100
BATCH     = 512
LR        = 1e-3

Loading and preprocessing data...
Dataset shape after preprocessing: (69990, 83)
Positive rate: 0.0898


In [1]:
 # ── 1. XGBoost baseline ──────────────────────────────────────────────────
print("\n\n>>> Running XGBoost baseline...")
xgb_results = train_kfold_xgb_baseline(df, save_dir=MODELS_DIR, force_retrain=FORCE_RETRAIN)
print_results("XGBoost Baseline", xgb_results)

# ── 2. Plain DNN ─────────────────────────────────────────────────────────
print("\n\n>>> Running DNN...")
dnn_results = train_kfold_dnn(
    df, DNN,
    num_epochs=N_EPOCHS,
    batch_size=BATCH,
    lr=LR,
    device=DEVICE,
    verbose=True,
    save_dir=MODELS_DIR,
    force_retrain=FORCE_RETRAIN,
)
print_results("DNN (plain, raw features)", dnn_results)




>>> Running XGBoost baseline...


NameError: name 'train_kfold_xgb_baseline' is not defined

In [3]:

# ── 3. TNT + Soft Decision Tree ──────────────────────────────────────────
print("\n\n>>> Running TNT + Soft Tree...")
tnt_sdt_results = train_kfold_TNT(
    df, TN_DNN,
    num_epochs=N_EPOCHS,
    batch_size=BATCH,
    lr=LR,
    device=DEVICE,
    distilation=True,
    verbose=True,
    save_dir=MODELS_DIR,
    force_retrain=True,
)
print_results("TNT + Soft Decision Tree", tnt_sdt_results)



>>> Running TNT + Soft Tree...

  Fold 1/5
Epoch [  1/100] Train BCE Loss: 1.2695  Val BCE Loss: 1.2552
Epoch [  2/100] Train BCE Loss: 1.2472  Val BCE Loss: 1.2369
Epoch [  3/100] Train BCE Loss: 1.2296  Val BCE Loss: 1.2246
Epoch [  4/100] Train BCE Loss: 1.2154  Val BCE Loss: 1.2191
Epoch [  5/100] Train BCE Loss: 1.2033  Val BCE Loss: 1.2195
Epoch [  6/100] Train BCE Loss: 1.1924  Val BCE Loss: 1.2151
Epoch [  7/100] Train BCE Loss: 1.1774  Val BCE Loss: 1.2212
Epoch [  8/100] Train BCE Loss: 1.1657  Val BCE Loss: 1.2237
Epoch [  9/100] Train BCE Loss: 1.1545  Val BCE Loss: 1.2252
Epoch [ 10/100] Train BCE Loss: 1.1416  Val BCE Loss: 1.2330
Epoch [ 11/100] Train BCE Loss: 1.1317  Val BCE Loss: 1.2329
Epoch [ 12/100] Train BCE Loss: 1.1195  Val BCE Loss: 1.2393
Epoch [ 13/100] Train BCE Loss: 1.1048  Val BCE Loss: 1.2593
Epoch [ 14/100] Train BCE Loss: 1.0930  Val BCE Loss: 1.2640
Epoch [ 15/100] Train BCE Loss: 1.0803  Val BCE Loss: 1.2684
Epoch [ 16/100] Train BCE Loss: 1.0670  

In [6]:

# ── 4. TNT + dGBDT ───────────────────────────────────────────────────────
print("\n\n>>> Running TNT + dGBDT...")
tnt_dgbdt_results = train_kfold_tnt_dgbdt(
    df, TN_DNN,
    num_epochs=N_EPOCHS,
    num_epochs_dgbdt=N_EPOCHS,
    batch_size=BATCH,
    lr=LR,
    lr_dgbdt=LR,
    device=DEVICE,
    alpha=0.5,
    temperature=1.0,
    dgbdt_num_trees=5,
    dgbdt_depth=5,
    dgbdt_lamda=1e-3,
    verbose=True,
    save_dir=MODELS_DIR,
    force_retrain=FORCE_RETRAIN,
)
print_results("TNT + dGBDT", tnt_dgbdt_results)




>>> Running TNT + dGBDT...

  Fold 1/5
Epoch [  1/100] Train BCE Loss: 1.2745  Val BCE Loss: 1.2315
Epoch [  2/100] Train BCE Loss: 1.2736  Val BCE Loss: 1.2310
Epoch [  3/100] Train BCE Loss: 1.2729  Val BCE Loss: 1.2307
Epoch [  4/100] Train BCE Loss: 1.2723  Val BCE Loss: 1.2303
Epoch [  5/100] Train BCE Loss: 1.2717  Val BCE Loss: 1.2299
Epoch [  6/100] Train BCE Loss: 1.2712  Val BCE Loss: 1.2296
Epoch [  7/100] Train BCE Loss: 1.2706  Val BCE Loss: 1.2293
Epoch [  8/100] Train BCE Loss: 1.2701  Val BCE Loss: 1.2289
Epoch [  9/100] Train BCE Loss: 1.2696  Val BCE Loss: 1.2286
Epoch [ 10/100] Train BCE Loss: 1.2690  Val BCE Loss: 1.2284
Epoch [ 11/100] Train BCE Loss: 1.2685  Val BCE Loss: 1.2281
Epoch [ 12/100] Train BCE Loss: 1.2680  Val BCE Loss: 1.2279
Epoch [ 13/100] Train BCE Loss: 1.2675  Val BCE Loss: 1.2276
Epoch [ 14/100] Train BCE Loss: 1.2669  Val BCE Loss: 1.2274
Epoch [ 15/100] Train BCE Loss: 1.2664  Val BCE Loss: 1.2273
Epoch [ 16/100] Train BCE Loss: 1.2659  Val 

In [7]:
# ── Summary ──────────────────────────────────────────────────────────────
print("\n\n" + "="*60)
print("  FINAL SUMMARY — AUROC (mean ± std)")
print("="*60)
print(f"  XGBoost Baseline   {xgb_results['mean_auroc']:.4f} ± {xgb_results['std_auroc']:.4f}")
print(f"  DNN                {dnn_results['mean_auroc']:.4f} ± {dnn_results['std_auroc']:.4f}")
print(f"  TNT Teacher        {tnt_sdt_results['mean_teacher_auroc']:.4f} ± {tnt_sdt_results['std_teacher_auroc']:.4f}")
print(f"  TNT + Soft Tree    {tnt_sdt_results['mean_soft_tree_auroc']:.4f} ± {tnt_sdt_results['std_soft_tree_auroc']:.4f}")
print(f"  TNT + dGBDT        {tnt_dgbdt_results['mean_dgbdt_auroc']:.4f} ± {tnt_dgbdt_results['std_dgbdt_auroc']:.4f}")
print("="*60)




  FINAL SUMMARY — AUROC (mean ± std)
  XGBoost Baseline   0.4959 ± 0.0655
  DNN                0.4879 ± 0.0980
  TNT Teacher        0.5607 ± 0.0370
  TNT + Soft Tree    0.5633 ± 0.0557
  TNT + dGBDT        0.5521 ± 0.0531
